## Task 12: Knowledge Distillation of Dual-Encoder Vision-Language Models
**Requires:** downloading a CLIP checkpoint (needs internet) and PyTorch/TorchVision to train a student network. Not available in this sandbox — reference implementation below.

In [1]:
!pip install torch torchvision transformers pillow -q

In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import CLIPModel, CLIPProcessor
from PIL import Image
import numpy as np

device = "cuda" if torch.cuda.is_available() else "cpu"

teacher = CLIPModel.from_pretrained("openai/clip-vit-base-patch32").to(device)
teacher.eval()
for p in teacher.parameters():
    p.requires_grad = False

processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")

class StudentEncoder(nn.Module):
    def __init__(self, out_dim=512):
        super().__init__()
        self.backbone = nn.Sequential(
            nn.Conv2d(3, 32, 3, stride=2, padding=1), nn.ReLU(),
            nn.Conv2d(32, 64, 3, stride=2, padding=1), nn.ReLU(),
            nn.AdaptiveAvgPool2d(1),
        )
        self.proj = nn.Linear(64, out_dim)

    def forward(self, x):
        feat = self.backbone(x).flatten(1)
        return self.proj(feat)

student = StudentEncoder().to(device)
optimizer = torch.optim.Adam(student.parameters(), lr=1e-4)

def distillation_loss(student_feat, teacher_feat, temperature=2.0):
    cos_loss = 1 - F.cosine_similarity(student_feat, teacher_feat).mean()
    kl_loss = F.kl_div(
        F.log_softmax(student_feat / temperature, dim=-1),
        F.softmax(teacher_feat / temperature, dim=-1),
        reduction='batchmean'
    )
    return 0.5 * cos_loss + 0.5 * kl_loss

def make_batch(batch_size=8):
    imgs = [Image.fromarray(np.random.randint(0, 255, (224, 224, 3), dtype=np.uint8)) for _ in range(batch_size)]
    inputs = processor(images=imgs, return_tensors="pt").to(device)
    return inputs["pixel_values"]

losses = []
for step in range(20):
    pixel_values = make_batch()

    with torch.no_grad():
        vision_outputs = teacher.vision_model(pixel_values=pixel_values)
        teacher_feat = teacher.visual_projection(vision_outputs.pooler_output)
        print("teacher_feat shape:", teacher_feat.shape)   # debug line — confirm it's 512

    student_feat = student(pixel_values)

    loss = distillation_loss(student_feat, teacher_feat)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    losses.append(loss.item())
    if step % 5 == 0:
        print(f"step {step}: loss = {loss.item():.4f}")

print("\nFinal loss:", round(losses[-1], 4))
print("Student output shape:", student_feat.shape, "| Teacher output shape:", teacher_feat.shape)

config.json:   0%|          | 0.00/4.19k [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  605MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

model.safetensors: reconstructing file:   0%|          |  0.00B /  605MB            

model.safetensors: downloading bytes:           |  0.00B            

...

vocab.json:   0%|          | 0.00/862k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/525k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.22M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/389 [00:00<?, ?B/s]

teacher_feat shape: torch.Size([8, 512])
step 0: loss = 0.4995
teacher_feat shape: torch.Size([8, 512])
teacher_feat shape: torch.Size([8, 512])
teacher_feat shape: torch.Size([8, 512])
teacher_feat shape: torch.Size([8, 512])
teacher_feat shape: torch.Size([8, 512])
step 5: loss = 0.4732
teacher_feat shape: torch.Size([8, 512])
teacher_feat shape: torch.Size([8, 512])
teacher_feat shape: torch.Size([8, 512])
teacher_feat shape: torch.Size([8, 512])
teacher_feat shape: torch.Size([8, 512])
step 10: loss = 0.4466
teacher_feat shape: torch.Size([8, 512])
teacher_feat shape: torch.Size([8, 512])
teacher_feat shape: torch.Size([8, 512])
teacher_feat shape: torch.Size([8, 512])
teacher_feat shape: torch.Size([8, 512])
step 15: loss = 0.4225
teacher_feat shape: torch.Size([8, 512])
teacher_feat shape: torch.Size([8, 512])
teacher_feat shape: torch.Size([8, 512])
teacher_feat shape: torch.Size([8, 512])

Final loss: 0.4045
Student output shape: torch.Size([8, 512]) | Teacher output shape: tor